In [7]:

import os
from pathlib import Path
import sys
import torch
import numpy as np

# use this if having problems:
# export PYTHONPATH=/home/ee577/project/src:$PYTHONPATH




In [5]:
pkg_path = str(Path(os.getcwd()).parent.absolute())  # Get parent directory of the current working directory
sys.path.insert(0, pkg_path)

# If you need to include a 'src' directory relative to the current notebook:
src_path = os.path.abspath(os.path.join(os.getcwd(), '../src'))  # Adjust path relative to the current working directory
sys.path.insert(0, src_path)

print(f"Path to the 'src' directory: {src_path}")

# Now you can import your modules from the 'src' directory
from src import *

# Load config file
config = global_config.config
config.device = 'cuda:1'
torch.manual_seed(config.seed)
print(f"This is the output directory: {config.dataset_dir}")

Path to the 'src' directory: /home/ee577/project/src
This is the output directory: /home/ee577/project/Datasets


In [ ]:
preprocess_config = {
    'modality': ['DTI'], #['T2', 'FLAIR', 'T1', 'T1GD']
    'image_type': 'autosegm',
    'window': (140, 172, 164),
    'pad_window': (70, 86, 86),
    'crop': True,
    'window_idx': ((0, 144), (31, 214), (44,209)),
    'down_factor': 0.5,
    'augments': ['base'] #'base', 'flip', 'rotate', 'noise', 'deform'
}

paths, mod_arr,tumor_boxes=data_prep.convert_image_data_mod(**preprocess_config)


In [25]:
first_image = list(mod_arr.values())[3]
print(f"Shape of the first image: {first_image.shape}")
first_key = list(mod_arr.keys())[3]
print(f"The first key is: {first_key}")

paths.head()

Shape of the first image: (74, 98, 86)
The first key is: TR


,"(0.0, 100.0]","(100.0, 200.0]","(200.0, 300.0]","(300.0, 400.0]","(400.0, 500.0]","(500.0, 700.0]","(700.0, inf]",autosegm_image_paths,mansegm_image_paths,dti_image_paths
ID,,,,,,,,,,
UPENN-GBM-00001_11,False,False,False,False,False,False,True,/home/ee577/project/Datasets/UPENN_GBM/PKG-UPE...,NaN,{'FA': '/home/ee577/project/Datasets/UPENN_GBM...
UPENN-GBM-00002_11,False,False,True,False,False,False,False,/home/ee577/project/Datasets/UPENN_GBM/PKG-UPE...,/home/ee577/project/Datasets/UPENN_GBM/PKG-UPE...,{'AD': '/home/ee577/project/Datasets/UPENN_GBM...
UPENN-GBM-00003_11,False,False,False,False,False,False,True,/home/ee577/project/Datasets/UPENN_GBM/PKG-UPE...,NaN,{'RD': '/home/ee577/project/Datasets/UPENN_GBM...
UPENN-GBM-00004_11,False,False,False,False,False,True,False,/home/ee577/project/Datasets/UPENN_GBM/PKG-UPE...,NaN,{'TR': '/home/ee577/project/Datasets/UPENN_GBM...
UPENN-GBM-00005_11,False,False,False,False,False,False,True,/home/ee577/project/Datasets/UPENN_GBM/PKG-UPE...,NaN,{'AD': '/home/ee577/project/Datasets/UPENN_GBM...


In [ ]:
import os
import numpy as np

def save_3d_images_from_paths(mod_arr, out_dir, paths):
    """
    Save 3D images from mod_arr dictionary to .npy files based on paths.

    Args:
        mod_arr (dict): Dictionary containing processed 3D images for each modality.
        out_dir (str): Output directory where images will be saved.
        paths (pd.DataFrame): DataFrame containing paths information for each patient and modality.

    Returns:
        None
    """
    # Ensure the output directory exists
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)

    # Iterate over patients in the DataFrame
    for pat, row in paths.iterrows():
        patient_id = pat  # Using patient ID from the DataFrame index
        
        # Iterate through each modality (based on the columns)
        for modality, image_arr in mod_arr.items():
            if image_arr is not None:  # If there is an image for the modality
                # Check if the modality exists in the paths DataFrame for this patient
                if row.get(f'{modality}_image_paths'):
                    # Get the output filename based on patient ID and modality
                    image_filename = f"{patient_id}_{modality}.npy"
                    image_filepath = os.path.join(out_dir, image_filename)
                    
                    # Save the 3D image as a .npy file
                    np.save(image_filepath, image_arr)
                    print(f"Saved 3D image for patient {patient_id}, modality {modality}: {image_filepath}")

# Example usage
out_dir = 'path/to/save/images'  # Specify your output directory where you want to save the images
save_3d_images_from_paths(mod_arr, out_dir, paths)


In [ ]:
# Example usage
out_dir = '/home/ee577/project/Datasets/UPENN_GBM/DTI_numpy_files'  # Specify your output directory where you want to save the images
save_3d_images(mod_arr, out_dir, paths)